# Robotic Knee & Hip Replacement – What FDA Reports Reveal
Case Study 4 · Dr. Muhammad Ali, DPT

**Question:** What problems are reported for robotic-assisted joint replacement systems (Stryker Mako, Zimmer Biomet ROSA, J&J DePuy VELYS), and which of them can better training and rehabilitation help prevent?

**Data:** FDA MAUDE (medical device adverse-event reports), via the openFDA API. All three robots are registered under FDA product code **OLO** ("orthopedic stereotaxic instrument").

**Gate logic:** every step ends with automatic checks ("gates"). If a number does not match the FDA's own count, the notebook stops with `GATE FAILED`. A `GATE PASSED` line is proof that the result matches the source data.

## Step 1 – Set up the project and check the FDA total
Create the project folders, define the gate check, and ask the FDA how many reports exist for product code OLO. This total is the reference number that every later step must match.

In [1]:
# 1.1 Set up folders, define the gate check, and get the FDA's total for product code OLO
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, requests
import pandas as pd

BASE  = '/content/drive/MyDrive/robotic-knee-fda'
RAW   = f'{BASE}/data/raw'
CLEAN = f'{BASE}/data/clean'
FIG   = f'{BASE}/figures'
for folder in [RAW, CLEAN, FIG]:
    os.makedirs(folder, exist_ok=True)

API   = 'https://api.fda.gov/device/event.json'
QUERY = 'device.device_report_product_code:OLO'

def gate(condition, message):
    """Stop the notebook if a check fails; print a pass line if it succeeds."""
    if not condition:
        raise AssertionError('GATE FAILED: ' + message)
    print('GATE PASSED:', message)

response = requests.get(API, params={'search': QUERY, 'limit': 1}, timeout=60)
gate(response.status_code == 200, 'FDA API answered (status 200)')

meta = response.json()['meta']
FDA_TOTAL = meta['results']['total']
print('FDA database last updated:', meta['last_updated'])
print('Reports with product code OLO:', FDA_TOTAL)
gate(FDA_TOTAL <= 25000, 'total is within the API download limit of 25,000 reports')

Mounted at /content/drive
GATE PASSED: FDA API answered (status 200)
FDA database last updated: 2026-09-22
Reports with product code OLO: 20634
GATE PASSED: total is within the API download limit of 25,000 reports


## Step 2 – Download every report and save the raw data
The openFDA API returns at most 1,000 reports per request, so the reports are downloaded in 21 pages and combined. The combined data is saved **untouched** as the raw file (`data/raw/olo_reports_raw.jsonl.gz`), so every later step can be traced back to exactly what the FDA published.

**Gates in this step:**
1. Downloaded reports = FDA total (nothing missing)
2. Every report key is unique (nothing downloaded twice)
3. Every report has a report key
4. The saved file, read back from Drive, still contains every report

In [3]:
# 2.1 Download every OLO report (100 per request), then save the raw data unchanged
import gzip

PAGE = 100
records = []
for skip in range(0, FDA_TOTAL, PAGE):
    for attempt in range(5):                       # retry if the FDA server is busy
        r = requests.get(API, params={'search': QUERY, 'limit': PAGE, 'skip': skip},
                         headers={'User-Agent': 'Mozilla/5.0 (research notebook)'}, timeout=120)
        if r.status_code == 200:
            break
        time.sleep(5 * (attempt + 1))
    if r.status_code != 200:
        print('FDA error message:', r.text[:500])   # shows the reason for the refusal
        raise RuntimeError(f'Download failed at skip={skip}, status {r.status_code}')
    records.extend(r.json()['results'])
    print(f'Downloaded {len(records):,} of {FDA_TOTAL:,}', end='\r')
    time.sleep(0.5)                                # stay well under the FDA rate limit
print()

keys = [rec.get('mdr_report_key') for rec in records]
gate(len(records) == FDA_TOTAL, f'downloaded reports ({len(records):,}) = FDA total ({FDA_TOTAL:,})')
gate(len(set(keys)) == len(keys), 'every report key is unique (no duplicates)')
gate(all(keys), 'every report has a report key')

raw_path = f'{RAW}/olo_reports_raw.jsonl.gz'
with gzip.open(raw_path, 'wt', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec) + '\n')

with gzip.open(raw_path, 'rt', encoding='utf-8') as f:     # read the file back to prove it saved fully
    saved = sum(1 for _ in f)
gate(saved == FDA_TOTAL, f'saved raw file contains all {saved:,} reports')
print(f'Raw file size: {os.path.getsize(raw_path) / 1e6:.1f} MB')


GATE PASSED: downloaded reports (20,634) = FDA total (20,634)
GATE PASSED: every report key is unique (no duplicates)
GATE PASSED: every report has a report key
GATE PASSED: saved raw file contains all 20,634 reports
Raw file size: 13.0 MB


## Step 3 – Turn the raw reports into a table and check it against the FDA
Each FDA report is a nested record. Here every report becomes **one row** with the fields needed for the analysis: date received, event type (death / injury / malfunction), manufacturer, brand name, reported device and patient problems, and the event description.

**Gates in this step:**
1. The table has exactly one row per report (= FDA total)
2. Report keys are still unique
3. Every report has a valid date
4. **Independent check:** the number of deaths, injuries and malfunctions in the table equals the FDA's own count, requested separately from the API

*Note: 13 reports have no event type recorded. The FDA counts them as a blank category, so they are kept and checked as "(blank)" rather than dropped.*


In [8]:
# 3.1 Flatten the reports into one row per report, then check the table against the FDA's own counts
if 'records' not in globals():                                   # reload the raw file if the notebook restarted
    with gzip.open(raw_path, 'rt', encoding='utf-8') as f:
        records = [json.loads(line) for line in f]

def join_values(values, sep='; '):
    """Join a list into text, skipping empty values (some FDA lists contain None)."""
    return sep.join(sorted({str(v).strip() for v in (values or []) if v not in (None, '')}))

def report_text(rec, kind):
    return ' '.join((t.get('text') or '') for t in (rec.get('mdr_text') or [])
                    if t.get('text_type_code') == kind).strip()

rows = []
for rec in records:
    devices = rec.get('device') or []
    rows.append({
        'report_key':       rec.get('mdr_report_key'),
        'date_received':    rec.get('date_received'),
        'event_type':       rec.get('event_type') or '',
        'manufacturer':     join_values([d.get('manufacturer_d_name') for d in devices], ' | '),
        'brand':            join_values([d.get('brand_name') for d in devices], ' | '),
        'product_problems': join_values(rec.get('product_problems')),
        'patient_problems': join_values([p for pt in (rec.get('patient') or []) for p in (pt.get('patient_problems') or [])]),
        'event_text':       report_text(rec, 'Description of Event or Problem'),
        'mfr_text':         report_text(rec, 'Additional Manufacturer Narrative'),
    })
df = pd.DataFrame(rows)
df['date_received'] = pd.to_datetime(df['date_received'], format='%Y%m%d', errors='coerce')

gate(len(df) == FDA_TOTAL, f'table has one row per report ({len(df):,})')
gate(df['report_key'].is_unique, 'report keys are unique in the table')
gate(df['date_received'].notna().all(), 'every report has a valid date')

# Independent check: ask the FDA for its own event-type counts and compare (blank event types included)
fda_counts = requests.get(API, params={'search': QUERY, 'count': 'event_type.exact'},
                          headers={'User-Agent': 'Mozilla/5.0 (research notebook)'}, timeout=60).json()['results']
fda_counts = {c['term']: c['count'] for c in fda_counts}
our_counts = df['event_type'].value_counts().to_dict()
gate(our_counts == fda_counts, 'death / injury / malfunction / other / blank counts match the FDA exactly')
gate(sum(our_counts.values()) == FDA_TOTAL, f'event-type groups add up to the full total ({FDA_TOTAL:,})')

print('\nEvent type     Our table   FDA')
for k in sorted(fda_counts, key=fda_counts.get, reverse=True):
    label = k if k else '(blank)'
    print(f'{label:<14} {our_counts.get(k, 0):>9,} {fda_counts[k]:>7,}')
print('\nDate range:', df['date_received'].min().date(), 'to', df['date_received'].max().date())

GATE PASSED: table has one row per report (20,634)
GATE PASSED: report keys are unique in the table
GATE PASSED: every report has a valid date
GATE PASSED: death / injury / malfunction / other / blank counts match the FDA exactly
GATE PASSED: event-type groups add up to the full total (20,634)

Event type     Our table   FDA
Malfunction       16,836  16,836
Injury             3,762   3,762
(blank)               13      13
Death                 13      13
Other                 10      10

Date range: 2008-10-20 to 2026-08-31


In [7]:
# Diagnostic: show both sets of counts side by side
print('FDA :', fda_counts)
print('Ours:', our_counts)
print('Sum FDA:', sum(fda_counts.values()), '| Sum ours:', sum(our_counts.values()),
      '| Reports with blank event type:', (df['event_type'] == '').sum())

FDA : {'Malfunction': 16836, 'Injury': 3762, '': 13, 'Death': 13, 'Other': 10}
Ours: {'Malfunction': 16836, 'Injury': 3762, 'Death': 13, 'Other': 10}
Sum FDA: 20634 | Sum ours: 20621 | Reports with blank event type: 13


## Step 4 – Pick out the three joint-replacement robots
Product code OLO also covers spine and brain navigation systems (e.g. Medtronic, Globus, Brainlab). Each report is assigned to **one** group using its brand and manufacturer names:

| Group | Rule |
|---|---|
| Stryker Mako | name contains "MAKO" |
| Zimmer Biomet ROSA | name contains the word "ROSA" |
| J&J VELYS | name contains "VELYS" |
| Other | everything else, including spine/brain versions ("ROSA SPINE", "ROSA ONE", "VELYS SPINE") |

**Gates:** no report matches more than one robot · the four groups add up to the FDA total · exactly four groups exist.
A random sample of brand names from each group is shown so the rules can be checked by eye.

In [9]:
# 4.1 Assign every report to one robotic system using its brand and manufacturer names
names = (df['brand'] + ' | ' + df['manufacturer']).str.upper()
spine_or_brain = names.str.contains(r'SPINE|ROSA ONE|BRAIN|CRANIAL')

flags = pd.DataFrame({
    'Stryker Mako':       names.str.contains(r'\bMAKO')     & ~spine_or_brain,
    'Zimmer Biomet ROSA': names.str.contains(r'\bROSA\b')   & ~spine_or_brain,
    'J&J VELYS':          names.str.contains(r'VELYS')      & ~spine_or_brain,
})
gate((flags.sum(axis=1) <= 1).all(), 'no report matches more than one robotic system')

OTHER = 'Other (spine, brain, other navigation)'
df['system'] = flags.idxmax(axis=1).where(flags.any(axis=1), OTHER)

counts = df['system'].value_counts()
gate(counts.sum() == FDA_TOTAL, f'the groups add up to the FDA total ({counts.sum():,})')
gate(set(counts.index) == set(flags.columns) | {OTHER}, 'exactly four groups exist')

excluded = (spine_or_brain & names.str.contains(r'\bMAKO|\bROSA\b|VELYS')).sum()
print('\nReports per group:')
print(counts.to_string())
print(f'\nSpine/brain versions of these brands moved to "Other": {excluded}')

print('\nSpot-check – 5 random brand names per robot:')
for s in flags.columns:
    sample = df.loc[df['system'] == s, 'brand'].drop_duplicates().sample(5, random_state=1)
    print(f'\n{s}:')
    for b in sample:
        print('  -', b[:90])

GATE PASSED: no report matches more than one robotic system
GATE PASSED: the groups add up to the FDA total (20,634)
GATE PASSED: exactly four groups exist

Reports per group:
system
Other (spine, brain, other navigation)    13095
Stryker Mako                               4984
J&J VELYS                                  2455
Zimmer Biomet ROSA                          100

Spine/brain versions of these brands moved to "Other": 117

Spot-check – 5 random brand names per robot:

Stryker Mako:
  - ROBOTIC ARM INTERACTIVE ORTHOPEDIC SYSTEM-THA
  - ROBOTIC ARM INTERATIVE ORTHOPEDIC SYSTEM-RIO
  - MAKO TKA SYSTEM
  - BONE PIN-3.2 X 80 MM-STERILE  (2 PACK)
  - USB TO FIBER CONVERTERS, LEX

Zimmer Biomet ROSA:
  - ROSA CUP INSERTER SCREW
  - ROSA KNEE SYSTEM
  - ROSA KNEE ROBOTIC UNIT US PL B
  - ROSA RECON PLATFORM 220V
  - ROSA KNEE PLATFORM GB PL G

J&J VELYS:
  - VELYS CAMERA STATION
  - VELYS
  - VELYS SAW INTERFACE LEFT
  - VELYS NAVIGATION STATION
  - VELYS ROBOTIC-ASSISTED SOLUTION SAW

### Step 4.2 – Check for missed reports
A report is only counted for a robot if its brand or manufacturer name contains MAKO, ROSA or VELYS. To make sure no robotic-joint reports were missed, this step lists the most common brand names that Zimmer, Stryker and DePuy (J&J) have in the "Other" group. Each name is then judged: is it a joint-replacement robot (should be moved), or a different product such as spine or brain navigation (correctly excluded)?

In [10]:
# 4.2 List brand names from Zimmer, Stryker and DePuy (J&J) that were left in "Other"
other = df[df['system'] == OTHER]
for company in ['ZIMMER', 'STRYKER', 'DEPUY']:
    subset = other[other['manufacturer'].str.upper().str.contains(company)]
    print(f'\n=== {company}: {len(subset):,} reports left in Other – top 12 brand names ===')
    print(subset['brand'].str.upper().value_counts().head(12).to_string())


=== ZIMMER: 210 reports left in Other – top 12 brand names ===
brand
IASSIST KNEE SYSTEM                                       22
CAS FIX PIN 3.2D X 150MM STR                              21
FIX PIN FLTD 3.2DX80MM 2 PACK                             15
CAS FIX PIN 3.2D X 80MM STR                               13
NAVIGATION CANNULATED T27 BONE SCREW DRIVER               12
VALIDATION TOOL                                           11
HEADLESS TROCAR DRILL PIN 3.2 MM DIAMETER 75 MM LENGTH    10
IASSIST KNEE                                               9
FIX PIN FLTD 3.2DX150MM 2 PACK                             6
UNKNOWN PIN                                                6
3.5MM HEX HEAD SCREW X38MM                                 6
SESAMOID PLASTY CAMERA KIT - P7                            6

=== STRYKER: 433 reports left in Other – top 12 brand names ===
brand
SPINEMAP® 3D 3.0 - SOFTWARE      44
ORTHO GRIP KNEE POINTER          32
SPINE GUIDANCE SOFTWARE          32
TIBIAL/PELVIC TRACK

**Result of the check:** none of the names left in "Other" belong to Mako, ROSA or VELYS, so the rule is kept unchanged.
- Zimmer: iASSIST Knee (handheld navigation, not a robot), fixation pins and screws
- Stryker: SpineMap / spine guidance software, and trackers for its non-robotic NAV3i navigation
- DePuy (J&J): navigated spine instruments (Expedium, Viper)

*Limitation:* generic accessories such as fixation pins or validation tools may be used with ROSA or with other navigation systems, but the report does not say which, so they cannot be attributed. The ROSA count (100) is therefore a minimum.

### Step 4.3 – Save the three-robot dataset
Only the reports for Stryker Mako, Zimmer Biomet ROSA and J&J VELYS are kept and saved as `data/clean/robot_reports.csv`. The file is read back from Drive and checked: total rows, rows per robot and unique report keys must all match the table above.

In [11]:
# 4.3 Keep only the three robots and save; read the file back to prove it is complete
robots = df[df['system'] != OTHER].copy()
expected = counts.drop(OTHER)

gate(len(robots) == expected.sum(), f'robot dataset has {len(robots):,} reports (Mako + ROSA + VELYS)')

out_path = f'{CLEAN}/robot_reports.csv'
robots.to_csv(out_path, index=False)

check = pd.read_csv(out_path, dtype={'report_key': str})
gate(len(check) == len(robots), 'saved file has the same number of rows')
gate(check['report_key'].is_unique, 'report keys are unique in the saved file')
gate(check['system'].value_counts().to_dict() == expected.to_dict(), 'rows per robot match exactly after saving')

print('\nSaved:', out_path)
print(check['system'].value_counts().to_string())

GATE PASSED: robot dataset has 7,539 reports (Mako + ROSA + VELYS)
GATE PASSED: saved file has the same number of rows
GATE PASSED: report keys are unique in the saved file
GATE PASSED: rows per robot match exactly after saving

Saved: /content/drive/MyDrive/robotic-knee-fda/data/clean/robot_reports.csv
system
Stryker Mako          4984
J&J VELYS             2455
Zimmer Biomet ROSA     100
